<div style="text-align: center;">
  <img src="../docs/imgs/banner.png" width="500">
</div>

# Fine-tuning with LoRA

### Imports and Model/Data Loading

In this class we will fine-tune [**`HuggingFaceTB/SmolLM2-360M-Instruct`**](https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct).

SmolLM2 is a **small transformer-based language model** with **360 million parameters**. Architecturally, it is similar to other modern LLMs (like GPT models): it uses a **decoder-only transformer**, meaning it predicts the next token in a sequence using self-attention over previous tokens.

You can think of it as a much smaller version of models like GPT or Llama.

Because the model is relatively small, it can fit comfortably in 16GB of VRAM, making it ideal for our session!

In [ ]:
import yaml
from dotenv import load_dotenv; load_dotenv()
from finetuning._utils import get_system_prompt, setup_hf; setup_hf()

from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import torch

from finetuning._defaults import WORKSHOP_DEFAULTS, ROOT_DIR
from finetuning._config import LoRAConfig

In [ ]:
# Load tokenizer and model (this will take a while the first time)

tokenizer = AutoTokenizer.from_pretrained(WORKSHOP_DEFAULTS.model)
model = AutoModelForCausalLM.from_pretrained(WORKSHOP_DEFAULTS.model)

For this section of the workshop we will fine-tune the model using a custom dataset:

[**`fingriffin/natural-questions-corporate-jargon`**](https://huggingface.co/datasets/fingriffin/natural-questions-corporate-jargon)

The dataset is derived from [**Google’s Natural Questions dataset**](https://huggingface.co/datasets/google-research-datasets/natural_questions), which is a large collection of real questions asked by users in Google Search. These questions cover a wide range of topics and are paired with factual answers sourced from Wikipedia.

For this exercise, the answers have been rewritten in exaggerated corporate jargon.

In other words, we are fine-tuning the model to respond to normal questions but using corporate style language.

### Chat Format

The dataset is structured using the **standard chat template** used by many modern instruction tuned models. Each training example is represented as a list of **messages**, where each message has:

- a **role** (such as `system`, `user` or `assistant`)
- **content** (the text of the message)

During training:

- The **user message** contains the question.
- The **assistant message** contains the corporate jargon answer.

In [ ]:
# Load dataset

ds = load_dataset(WORKSHOP_DEFAULTS.dataset)

ds

In [ ]:
# Let's pick out a random example

row = ds['train'][37]

user = row['messages'][1]
print(user, "\n")

assistant = row['messages'][2]
assistant

### LoRA Configuration Loading

We are now going to load the configuration for our LoRA experiment. The hyperparameters are set in a `yaml` configuration file, an example of which can be found under `configs/example_lora.yaml`. We encourage everyone to experiment with these parameters to produce different results in their own training process, but we place some restrictions on possible values to keep computations tractable on the GPUs we are working with. The fields are as follows:

- **num_epochs**: number of epochs for training. *Allowed range: [1, 3]*.
- **learning_rate**: learning rate for training. *Allowed range: [0, ∞)*.
- **lora_rank**: rank of the LoRA adapter to be trained. *Allowed range: [1, 32]*.
- **lora_alpha**: scaling parameter ('alpha') for base weight updates. *Allowed range: [1, ∞)*
- **micro_batch_size**: number of training examples processed by a single GPU in one forward/backward pass.
- **gradient_accumulation_steps**: number of mini batches processed before LoRA adapter weights are updated.
    - The product of **micro_batch_size** and **gradient_accumulation_steps** is referred to as the **global_batch_size**, which has an allowed range *[1, 4]*.

We use a `pydantic.BaseModel` subclass to enforce these safe ranges:

In [ ]:
CONFIG_PATH = ROOT_DIR / "configs" / "example_lora.yaml" # Change example_lora.yaml to your config file, or update it in place!

with open(CONFIG_PATH) as f:
    data = yaml.safe_load(f)

config = LoRAConfig(**data)

config

When building your training loop, you'll access hyperparameters via the config object we've just instantiated, e.g.

In [ ]:
config.learning_rate

### Training

Let's start by talking to our base model. See if you can change the system prompt to achieve better results.

In [ ]:
SYSTEM_PROMPT = 'You are a helpful assistant. Respond using as much corporate jargon as possible, while still retaining factual accuracy.'
USER_MESSAGE = 'Is pink rock salt the same as sea salt?'

# Construct messages
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_MESSAGE},
]

# Tokenize with the chat template
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
)

# Move tensors to the model device
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=WORKSHOP_DEFAULTS.max_new_tokens,
        temperature=WORKSHOP_DEFAULTS.temperature,
        do_sample=WORKSHOP_DEFAULTS.do_sample,
        pad_token_id=tokenizer.eos_token_id,
    )

# Decode only the newly generated tokens
prompt_length = inputs["input_ids"].shape[-1]
response = tokenizer.decode(
    outputs[0][prompt_length:],
    skip_special_tokens=True,
)

response

Now let's build the LoRA fine-tuning pipeline step by step. The aim here is to make the mechanics of training as visible as possible and the sample solutions for the below exercises can be found in [`notebooks/solutions/lora_solutions.ipynb`](solutions/lora_solutions.ipynb).

We will:
1. Prepare a small preprocessing function for chat examples
2. Wrap the base model with LoRA adapters
3. Build a dataloader and optimizer
4. Write a minimal training loop with gradient accumulation
5. Compare generations before and after training

#### 1. Preprocess the dataset

Each dataset row already contains a `messages` list. We want to:
- Turn those messages into one chat formatted string
- Tokenize it
- Truncate long examples to a sensible length

Fill in the missing pieces below.

In [ ]:
from peft import LoraConfig as PeftLoraConfig, TaskType, get_peft_model
from transformers import DataCollatorForLanguageModeling
from torch.utils.data import DataLoader

MAX_LENGTH = 512

def preprocess(example):
    # TODO: convert example["messages"] into a chat formatted string
    text = ________________________________

    # TODO: tokenize the text with truncation and max_length=MAX_LENGTH
    tokens = ________________________________

    return tokens


# TODO: map preprocess over ds["train"]
train_ds = ________________________________

# TODO: remove the original columns once tokenized
train_ds = ________________________________

train_ds[0]

#### 2. Attach LoRA adapters

Now wrap the base model with LoRA using the hyperparameters from `config`.

After this, only a small subset of weights should be trainable.

In [ ]:
peft_config = PeftLoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=________________,
    lora_alpha=________________,
    target_modules=________________,
    bias="none",
)

# TODO: wrap the base model with LoRA adapters
model = ________________________________

model.print_trainable_parameters()

#### 3. Build the dataloader and optimizer

We will use a standard causal language model data collator, and AdamW for optimisation.

In [ ]:
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# TODO: build the dataloader using the micro batch size from config
train_loader = ________________________________

# TODO: create the optimizer with config.learning_rate
optimizer = ________________________________

len(train_loader)

#### 4. Write the training loop

This is the key exercise. Remember that gradient accumulation lets us simulate a larger effective batch size:

`global_batch_size = micro_batch_size × gradient_accumulation_steps`

Fill in the blanks below.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.train()
optimizer.zero_grad()

loss_history = []

global_step = 0
for epoch in range(config.num_epochs):
    for step, batch in enumerate(train_loader, start=1):
        # TODO: move every tensor in the batch onto the device
        batch = ________________________________

        # TODO: run the forward pass
        outputs = ________________________________

        # TODO: get the loss and scale it for gradient accumulation
        loss = ________________________________

        # TODO: backpropagate
        ________________________________

        if step % config.gradient_accumulation_steps == 0 or step == len(train_loader):
            # TODO: update parameters and reset gradients
            ________________________________
            ________________________________
            global_step += 1

        loss_history.append(loss.item() * config.gradient_accumulation_steps)

        if step % 10 == 0 or step == len(train_loader):
            print(
                f"epoch={epoch + 1} step={step}/{len(train_loader)} "
                f"loss={loss_history[-1]:.4f}"
            )

---
Congratulations, you have just fine-tuned an LLM!

Now, let's see how our model does on an unseen example. Have some fun experimenting with different questions!

In [ ]:
QUESTION = "What is LoRA fine-tuning?"

# Construct messages
# Note: we must use the same system prompt as that used during training
messages = [
    {"role": "system", "content": get_system_prompt(ds)},
    {"role": "user", "content": QUESTION},
]

model.eval()

inputs = tokenizer(messages, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=WORKSHOP_DEFAULTS.max_new_tokens,
        temperature=WORKSHOP_DEFAULTS.temperature,
        do_sample=WORKSHOP_DEFAULTS.do_sample,
        pad_token_id=tokenizer.eos_token_id,
    )

finetuned_response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)

finetuned_response

And let's evaluate the loss on the test set:

In [ ]:
test_ds = ds["test"].map(preprocess)
test_ds = test_ds.remove_columns(ds["test"].column_names)

test_loader = DataLoader(
    test_ds,
    batch_size=config.micro_batch_size,
    shuffle=False,
    collate_fn=collator,
)

model.eval()
total_loss = 0.0

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        total_loss += outputs.loss.item()

print(f"Test loss: {total_loss / len(test_loader):.4f}")